# VisClick — Part C (step 2): RICO, Zenodo unified bundle, VINS

**Prerequisite:** step 1 finished (`01_pull_and_data.ipynb`) — CLAY + UI-Vision; **RICO** optional (download/extract for `unique_uis.tar.gz` in `01`).

**This notebook:** mount Drive → **pull** → **RICO** (skips if `data/rico/` already filled from `01`) → **Zenodo** bundle → **VINS** clone. **WebUI raw** optional — see `VisClick_Detailed_Plan.md` C.2.3 / C.2.6.

**Colab session:** you do **not** need to keep the previous runtime. Each notebook is self-contained (mount + `git pull`).

**Report:** every step prints `REPORT ...` lines. Run the **final** code cell and copy **all** `REPORT` lines to your notes — they map to `VisClick_Report_Data_Form.md` **§0**.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
ROOT = "/content/visclick"
if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    print("Cloned to", ROOT)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", ROOT, "pull", "--rebase", "origin", "main"], check=False)
    print("Pulled latest in", ROOT)
print("REPORT git_head =", subprocess.check_output(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
%cd /content/visclick

In [ ]:
import os
DRIVE = "/content/drive/MyDrive/visclick"
for sub in (
    "data/raw", "data/clay", "data/vins", "data/webui", "data/rico", "data/ui_vision",
    "data/unified", "data/source_train", "data/desktop_labeled", "data/desktop_test",
    "weights/baseline_source", "weights/experiments", "videos",
):
    os.makedirs(os.path.join(DRIVE, sub), exist_ok=True)
print("DRIVE =", DRIVE)
print("REPORT DRIVE_ROOT =", DRIVE)

In [ ]:
!pip install -q tqdm
import os, sys, subprocess
DRIVE = "/content/drive/MyDrive/visclick"
RAW = os.path.join(DRIVE, "data", "raw")
os.makedirs(RAW, exist_ok=True)

## C.2.1 — RICO (screenshots + view hierarchies)

If RICO is **already in `data/rico/`** from `01`, the next cell **skips**. Otherwise it looks for `data/raw/unique_uis.tar.gz` (or rico `*.zip`) and extracts — or run the wget cell in `01` first. Official URL: <https://storage.googleapis.com/crowdstf-rico-uiuc-4540/rico_dataset_v0.1/unique_uis.tar.gz>.


In [ ]:
import os, subprocess
DRIVE = "/content/drive/MyDrive/visclick"
raw = os.path.join(DRIVE, "data", "raw")
rico_dir = os.path.join(DRIVE, "data", "rico")
os.makedirs(rico_dir, exist_ok=True)
t_official = os.path.join(raw, "unique_uis.tar.gz")
t_alt = os.path.join(raw, "rico_unique_uis.tar.gz")
rico_zip = os.path.join(raw, "rico_unique_uis.zip")

def _rico_has_files(p):
    # os.walk is recursive: files under data/rico/combined/ count when p is data/rico
    if not os.path.isdir(p):
        return False
    for r, _d, files in os.walk(p):
        if files:
            return True
    return False

if _rico_has_files(rico_dir):
    print("RICO already present. Skipping extract. Path:", rico_dir)
    print("REPORT step = RICO | status = ALREADY_EXTRACTED | path =", rico_dir)
elif os.path.isfile(t_official) or os.path.isfile(t_alt):
    t = t_official if os.path.isfile(t_official) else t_alt
    r = subprocess.run(["tar", "-xzf", t, "-C", rico_dir], capture_output=True)
    print("RICO tar exit", r.returncode, "->", rico_dir, "(from", t, ")")
    print("REPORT step = RICO | status = EXTRACTED | method = tar | exit =", r.returncode, "| path =", rico_dir)
elif os.path.isfile(rico_zip):
    r = subprocess.run(["unzip", "-o", "-q", rico_zip, "-d", rico_dir], capture_output=True)
    print("RICO unzip exit", r.returncode, "->", rico_dir)
    print("REPORT step = RICO | status = EXTRACTED | method = zip | exit =", r.returncode, "| path =", rico_dir)
else:
    print("No archive in", raw, "— expect unique_uis.tar.gz or rico_unique_uis.zip. Run 01 C.2.1 first.")
    print("REPORT step = RICO | status = MANUAL_PENDING | raw_dir =", raw)

## C.2.6 — Zenodo: unified RICO + WebUI (YOLO format)

Record: <https://zenodo.org/records/19195885> — large download (~several GB). Uses `wget -c` so you can re-run if Colab disconnects.

Extracts to `data/unified/`. If **`data/unified/` already has files** (from a past run), download + unzip are **skipped** — you may **delete** `data/raw/rico_webui_unified.zip` after a good extract to save space. If the zip is missing and `unified/` is empty, the next cell runs `wget` again.


In [ ]:
import os, subprocess
DRIVE = "/content/drive/MyDrive/visclick"
raw = os.path.join(DRIVE, "data", "raw")
unified_root = os.path.join(DRIVE, "data", "unified")
os.makedirs(raw, exist_ok=True)
os.makedirs(unified_root, exist_ok=True)
zip_path = os.path.join(raw, "rico_webui_unified.zip")
url = "https://zenodo.org/records/19195885/files/dataset.zip?download=1"

def _unified_has_files(p):
    if not os.path.isdir(p):
        return False
    for r, _d, files in os.walk(p):
        if files:
            return True
    return False

if _unified_has_files(unified_root):
    print("data/unified/ already has data — skipping Zenodo download/unzip. (Safe to keep zip deleted.)")
    print("REPORT step = ZENODO_UNIFIED | status = ALREADY_EXTRACTED | path =", unified_root)
elif not os.path.isfile(zip_path):
    r_wget = subprocess.run(["wget", "-c", "-O", zip_path, url], check=False)
    print("Zenodo wget exit", r_wget.returncode, "->", zip_path)
    print("REPORT step = ZENODO_UNIFIED | download_exit =", r_wget.returncode, "| path =", zip_path)
else:
    print("Using existing", zip_path)
    print("REPORT step = ZENODO_UNIFIED | download = SKIPPED_EXISTS | path =", zip_path)
if not _unified_has_files(unified_root) and os.path.isfile(zip_path) and os.path.getsize(zip_path) > 1_000_000:
    r_unzip = subprocess.run(["unzip", "-o", "-q", zip_path, "-d", unified_root], capture_output=True)
    print("unzip exit", r_unzip.returncode, "->", unified_root)
    print("REPORT step = ZENODO_UNIFIED | unzip_exit =", r_unzip.returncode, "| extract_dir =", unified_root)
elif not _unified_has_files(unified_root):
    print("Zip missing or too small — fix download, then re-run this cell.")
    print("REPORT step = ZENODO_UNIFIED | status = INCOMPLETE_ZIP")

## C.2.4 — VINS (repo + external images)

Clones the **code** repo to `data/raw/vins_repo/`. The **image archive** is separate — follow the [VINS README](https://github.com/sbunian/VINS) and place/extract under `data/vins/` when you have it, then re-run the REPORT cell at the end.


In [ ]:
import os, subprocess
DRIVE = "/content/drive/MyDrive/visclick"
raw = os.path.join(DRIVE, "data", "raw")
os.makedirs(raw, exist_ok=True)
vins_repo = os.path.join(raw, "vins_repo")
if not os.path.isdir(os.path.join(vins_repo, ".git")):
    r = subprocess.run(["git", "clone", "https://github.com/sbunian/VINS.git", vins_repo], check=False)
    print("VINS git clone exit", r.returncode, "->", vins_repo)
    print("REPORT step = VINS_REPO | exit =", r.returncode, "| path =", vins_repo)
else:
    subprocess.run(["git", "-C", vins_repo, "pull"], check=False)
    print("VINS repo already present; pulled latest. Path:", vins_repo)
    print("REPORT step = VINS_REPO | status = EXISTS | path =", vins_repo)
print("If images are not in data/vins/ yet, download per VINS README, then re-run the final REPORT cell.")

## C.2.3 — WebUI raw (optional)

The Zenodo **unified** bundle in C.2.6 is usually enough for training. Raw Web-7k is only if you need the original site dumps — not automated here. Mark as SKIPPED in the report unless you add it later.

- Project: <https://uimodeling.github.io/>


In [ ]:
print("REPORT step = WEBUI_RAW | status = OPTIONAL_SKIPPED (use Zenodo unless you have a reason)")

## Full REPORT block (copy everything below the line for `VisClick_Report_Data_Form.md` §0)


In [ ]:
import os, sys, subprocess
DRIVE = "/content/drive/MyDrive/visclick"

def _dir_stats(p):
    n, total = 0, 0
    if not os.path.isdir(p):
        return 0, 0
    for r, _d, files in os.walk(p):
        for f in files:
            fp = os.path.join(r, f)
            try:
                total += os.path.getsize(fp)
                n += 1
            except OSError:
                pass
    return n, total

def _file_size(p):
    try:
        return os.path.getsize(p) if os.path.isfile(p) else 0
    except OSError:
        return 0

paths = {
    "RICO_EXTRACTED": os.path.join(DRIVE, "data", "rico"),
    "RICO_TAR": os.path.join(DRIVE, "data", "raw", "unique_uis.tar.gz"),
    "RICO_ZIP": os.path.join(DRIVE, "data", "raw", "rico_unique_uis.zip"),
    "ZENODO_ZIP": os.path.join(DRIVE, "data", "raw", "rico_webui_unified.zip"),
    "UNIFIED": os.path.join(DRIVE, "data", "unified"),
    "VINS_REPO": os.path.join(DRIVE, "data", "raw", "vins_repo"),
    "VINS_IMAGES": os.path.join(DRIVE, "data", "vins"),
    "CLAY": os.path.join(DRIVE, "data", "clay"),
    "UI_VISION": os.path.join(DRIVE, "data", "ui_vision"),
}

print("REPORT form_section = §0")
print("REPORT notebook = 02_rico_zenodo_vins")
print("REPORT colab_python =", sys.version.split()[0])
try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10,
    )
    print("REPORT §1.2 nvidia_smi =", (r.stdout or "") .strip() .replace(chr(10), " | ") or "NO_OUTPUT")
except FileNotFoundError:
    print("REPORT §1.2 nvidia_smi = NOT_INSTALLED")
for label, p in paths.items():
    if label.endswith("_ZIP") or label.endswith("_TAR"):
        b = _file_size(p)
        st = "OK" if b > 0 else "MISSING"
        print(f"REPORT dataset = {label} | path = {p} | file_bytes = {b} | size_GB = {b / (1024**3):.3f} | status = {st}")
    else:
        nf, b = _dir_stats(p)
        st = "EMPTY" if nf == 0 else "OK"
        print(f"REPORT dataset = {label} | path = {p} | files = {nf} | size_bytes = {b} | size_GB = {b / (1024**3):.3f} | status = {st}")
try:
    import importlib
    for m in ("torch", "numpy"):
        try:
            v = importlib.import_module(m).__version__
            print(f"REPORT §1.3 {m} = {v}")
        except Exception as e:
            print(f"REPORT §1.3 {m} = NOT_INSTALLED", e)
except Exception as e:
    print("REPORT §1.3 import_note =", e)
print("REPORT end")

## Stop here (step 2)

Confirm in Drive: `data/rico/` and/or RICO zip, `data/unified/`, `data/raw/vins_repo/`, and (when you have it) `data/vins/` images.

**Next (later):** EDA + `source_train` assembly (plan C.4, C.5) — a **third** notebook when you are ready, same pattern (mount, pull, `REPORT` lines).
